# Assignment 2: Retrieval-Augmented Generation Question Answering

Welcome to the second assignment for 50.055 Machine Learning Operations This is a continuation of your work in assignment 1, where you will be working to build a chatbot which can answer questions about SUTD to prospective students.


**This assignment is a individual assignment.**

- Read the instructions in this notebook carefully
- Add your solution code and answers in the appropriate places. The questions are marked as **QUESTION:**, the places where you need to add your code and text answers are marked as **ADD YOUR SOLUTION HERE**
- The completed notebook, including your added code and generated output will be your submission for the assignment.
- The notebook should execute without errors from start to finish when you select "Restart Kernel and Run All Cells..". Please test this before submission.
- Use the SUTD Education Cluster to solve and test the assignment. If you work on another environment, minimally test your work on the SUTD Education Cluster.

**Rubric for assessment** 

Your submission will be graded using the following criteria. 
1. Code executes: your code should execute without errors. The SUTD Education cluster should be used to ensure the same execution environment.
2. Correctness: the code should produce the correct result or the text answer should state the factual correct answer.
3. Style: your code should be written in a way that is clean and efficient. Your text answers should be relevant, concise and easy to understand.
4. Partial marks will be awarded for partially correct solutions.
5. Creativity and innovation: in this assignment you have more freedom to design your solution, compared to the first assignments. You can show of your creativity and innovative mindset. 
6. There is a maximum of 215 points for this assignment, which will be normalized to 10% of your grade.

**ChatGPT policy** 

If you use AI tools, such as ChatGPT, Claude, Gemini, etc. to solve the assignment questions, you need to be transparent about its use and mark AI-generated content as such. In particular, you should include the following in addition to your final answer:
- A copy or screenshot of the prompt you used
- The name of the AI model
- The AI generated output
- An explanation why the answer is correct or what you had to change to arrive at the correct answer

**Assignment Notes:** Please make sure to save the notebook as you go along. Submission instructions are located at the bottom of the notebook.



### Retrieval-Augmented Generation (RAG) 

In this assignment, you will be building a Retrieval-Augmented Generation (RAG) question answering system which can answer questions about SUTD.

We'll be leveraging `langchain` and the LLM API of your choice from assignment 1

Check out the docs:
- [LangChain](https://docs.langchain.com/docs/)


The SUTD website used to allow chatting with current students. Unfortunately, this feature does not exist anymore. Let's build a chatbot to fill this gap!


# Install dependencies
Use pip to install all required dependencies of this assignment in the cell below. Make sure to test this on the SUTD cluster as different environments have different software pre-installed.  

Use of AI is documented and attached in a link here as I do not want to mess up the notebook:

link.com

In [21]:
# QUESTION: Install and import all required packages
# The rest of your code should execute without any import or dependency errors.

# **--- ADD YOUR SOLUTION HERE (10 points) ---**

%pip install -q requests beautifulsoup4 pandas

# this was ran aft chunking
%pip install -q langchain langchain-community sentence-transformers faiss-cpu transformers torch accelerate

# ran before llm
%pip install -q openai python-dotenv

# ran before putting it all tgt
%pip install -q langchain-openai

# ----------------


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [22]:
import requests, bs4, pandas


# Download documents
The RAG application should be able to answer questions based on ingested documents. For the SUTD chatbot, download PDF and HTML files from the SUTD website. The documents should contain information about the admission process, available courses and the university in general.


As my initial assignment 1 dataset only had 5 documents, I had to supplement it with more. I re-used the exact same .py code from assignment 1 to scrape the website and convert it from html to text. The process involved adding the website links to `data/seed_urls_pages.txt` and then running `src/fetch_html.py` and `src/extract_page_content.py`. However if you were to just run the cells below again while grading, the sites will be re-scraped and re-added causing duplicates. Hence I introduced these toggles below. For now, it is set to false as all files have been downloaded and processed.

In [47]:
# QUESTION: Download documents from the SUTD website
# You should download at least 10 documents but more documents can increase the knowledge base of your chatbot.

# **--- ADD YOUR SOLUTION HERE (20 points) ---**

RUN_SCRAPING = False
RUN_EXTRACTION = False


In [24]:
from pathlib import Path

if RUN_SCRAPING:
    !python -m src.fetch_html \
      --seed data/seed_urls_pages.txt \
      --out data/raw/pages \
      --meta data/raw/pages/metadata.csv \
      --delay 2
else:
    print("Skipping HTML fetching (RUN_SCRAPING=False)")

raw_files = list(Path("data/raw/pages").glob("*.html"))

print(f"Total HTML files fetched: {len(raw_files)}\n")

Skipping HTML fetching (RUN_SCRAPING=False)
Total HTML files fetched: 22



In [25]:
if RUN_EXTRACTION:
    !python -m src.extract_page_content \
      --raw data/raw/pages \
      --out data/processed/docs \
      --meta data/processed/docs/metadata.csv
else:
    print("Skipping content extraction (RUN_EXTRACTION=False)")

docs = list(Path("data/processed/docs").rglob("*.txt"))

print(f"Total cleaned documents created: {len(docs)}\n")

Skipping content extraction (RUN_EXTRACTION=False)
Total cleaned documents created: 23



# Split documents
Use LangChain to split the documents into smaller text chunks. 

In [26]:
# building registry first

from pathlib import Path
import pandas as pd

raw_meta = pd.read_csv("data/raw/pages/metadata.csv")

raw_meta["raw_file"] = raw_meta["raw_path"].apply(lambda x: Path(x).name if pd.notna(x) else "")
raw_lookup = raw_meta.set_index("raw_file")[["url", "retrieved_at"]].to_dict("index")

rows = []

base = Path("data/processed/docs")

for domain_dir in ["academics", "admissions", "housing"]:
    folder = base / domain_dir

    for fp in sorted(folder.glob("*_content.txt")):
        raw_file = fp.name.replace("_content.txt", ".html")
        src = raw_lookup.get(raw_file, {})
        text = fp.read_text(encoding="utf-8")
        rows.append({
            "doc_id": fp.stem,
            "domain": domain_dir,
            "path": str(fp),
            "source_url": src.get("url", ""),
            "retrieved_at": src.get("retrieved_at", ""),
            "text": text
        })

doc_registry = pd.DataFrame(rows)

print("Total documents:", len(doc_registry))
doc_registry.head()

Total documents: 22


,doc_id,domain,path,source_url,retrieved_at,text
0,about_design-ai_content,academics,data/processed/docs/academics/about_design-ai_...,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...
1,asd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/asd_education_un...,https://www.sutd.edu.sg/asd/education/undergra...,2026-03-14T09:15:06.623893+00:00,Education\nUndergraduate programme\nASD curric...
2,dai_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/dai_education_un...,https://www.sutd.edu.sg/dai/education/undergra...,2026-03-14T09:15:10.901466+00:00,Education\nUndergraduate programme\nDAI curric...
3,education_undergraduate_content,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduate,2026-02-24T16:53:09.057783+00:00,Design·AI Education\nUndergraduate studies at ...
4,education_undergraduate_freshmore-subjects_con...,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduat...,2026-02-24T16:53:11.186665+00:00,Design·AI Education\nUndergraduate studies at ...


I then displayed some stats in order to make a better informed decision on how and what parameters for chunking.

In [27]:
# doc length statistics (shows smallest, biggest and avg)
doc_registry["length"] = doc_registry["text"].apply(len)

print(doc_registry["length"].describe())

count       22.000000
mean      6139.590909
std       4342.285581
min       1172.000000
25%       2446.500000
50%       4991.500000
75%       8947.500000
max      15794.000000
Name: length, dtype: float64


In [28]:
doc_registry[["doc_id", "length"]].sort_values("length", ascending=False).head(5)


,doc_id,length
6,esd_education_undergraduate_curriculum_beyond-...,15794
2,dai_education_undergraduate_curriculum_content,12541
7,istd_education_undergraduate_curriculum_content,12439
1,asd_education_undergraduate_curriculum_content,11468
14,admissions_undergraduate_education-expenses_fe...,10282


In [29]:

# QUESTION: Use langchain to split the documents into chunks 

#--- ADD YOUR SOLUTION HERE (20 points)---


from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

documents = [
    Document(
        page_content=row["text"],
        metadata={
            "doc_id": row["doc_id"],
            "domain": row["domain"],
            "source_url": row["source_url"],
            "retrieved_at": row["retrieved_at"],
            "path": row["path"],
        }
    )
    for _, row in doc_registry.iterrows()
]

# langchain text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, ### initial: 1000
    chunk_overlap=150, ### initial: 150
    separators=["\n\n", "\n", " ", ""]
)
chunked_docs = text_splitter.split_documents(documents)

# make into table
chunk_rows = []
for i, doc in enumerate(chunked_docs):
    chunk_rows.append({
        "chunk_id": f"{doc.metadata['doc_id']}_chunk_{i:03d}",
        "doc_id": doc.metadata["doc_id"],
        "domain": doc.metadata["domain"],
        "source_url": doc.metadata["source_url"],
        "retrieved_at": doc.metadata["retrieved_at"],
        "path": doc.metadata["path"],
        "text": doc.page_content,
        "chunk_length": len(doc.page_content),
    })

chunk_df = pd.DataFrame(chunk_rows)

print("Total documents:", len(documents))
print("Total chunks:", len(chunked_docs))
chunk_df.head(3)

Total documents: 22
Total chunks: 140


,chunk_id,doc_id,domain,source_url,retrieved_at,path,text,chunk_length
0,about_design-ai_content_chunk_000,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...,967
1,about_design-ai_content_chunk_001,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,By going beyond its traditional role as a tech...,846
2,about_design-ai_content_chunk_002,about_design-ai_content,academics,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00,data/processed/docs/academics/about_design-ai_...,SUTD’s Design·AI principle puts the human back...,768


In [30]:
# QUESTION: create embeddings of document chunks and store them in a local vector store for fast lookup
# Decide an appropriate embedding model. Use Huggingface to run the embedding model locally.
# You do not have to use cloud-based APIs.

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

langchain_docs = chunked_docs

print(f"Total LangChain documents: {len(langchain_docs)}")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(langchain_docs, embedding_model)

print("FAISS vector store created successfully.")
print(f"Total chunks stored: {len(langchain_docs)}")

vectorstore.save_local("data/vectorstore/faiss_index")
print("Vector store saved to data/vectorstore/faiss_index")


Total LangChain documents: 140


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8370.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store created successfully.
Total chunks stored: 140
Vector store saved to data/vectorstore/faiss_index


### QUESTION: 

What chunking method or strategy did you use? Why did you use this method. Explain your design decision in less than 10 sentences.


**--- ADD YOUR SOLUTION HERE (10 points) ---**


I chose Langchain's `RecursiveCharacterTextSplitter` because it tries to split the text at natural positions like paragraphs and spaces so this makes the chunks more readable and meaningful.
After displaying the stats, the pages vary in length, with minimum size of 1200 chars, mean of 6.1k and max of 15.8k. I initially went with a chunk size of 1000 but the results were not good so I retried at 1200 hoping a larger chunk would allow more context in each chunk for the model to understand and the results improved. Overlap is at 150 which is within the 10-20% range of the chunk size, and this ensures information near the chunk boundaries are not lost. This is extra helpful for website text where headers, requirements and important information can spank across adjacent chunks.



------------------------------


### QUESTION: 

What embeddings and vector store did you use and why? Explain your design decision in less than 10 sentences.

**--- ADD YOUR SOLUTION HERE (10 points) ---**


Embedding model: `sentence-transformers/all-MiniLM-L6-v2`

Vector store: `FAISS`

I chose this embedding model because it is lightweight, fast, and widely used for semantic search, which makes it suitable for my very small local RAG pipeline. It can run locally without cloud API. The model converts each chunk into a vector so that similar chunks can be retreieved even when the wording of the question is different from the exact text in the documents stored.

I used FAISS because it is efficient for similarity search and integrates well with Langchain. Since my dataset is small, this vector store is sufficient and gives fast local retrieval.

Overall, this combination provided a practical balance of speed / ease of use (ran locally) and retrieval quality.

------------------------------



In [32]:
# Execute a query against the vector store

query = "When was SUTD founded?"

# QUESTION: run the query against the vector store, print the top 5 search results

results = vectorstore.similarity_search(query, k=5)

print(f"Top {len(results)} results for query: {query}\n")

for i, doc in enumerate(results, start=1):
    print("=" * 120)
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:800])
    print()

Top 5 results for query: When was SUTD founded?

Result 1
Metadata: {'doc_id': 'media-releases-listing_sutd-broadens-scope-of-flagship-design-and-ai-degree-first-university-to-integrate-social-sciences-into-technology-degree_content', 'domain': 'academics', 'source_url': 'https://www.sutd.edu.sg/media-releases-listing/sutd-broadens-scope-of-flagship-design-and-ai-degree-first-university-to-integrate-social-sciences-into-technology-degree', 'retrieved_at': '2026-03-14T08:56:47.465744+00:00', 'path': 'data/processed/docs/academics/media-releases-listing_sutd-broadens-scope-of-flagship-design-and-ai-degree-first-university-to-integrate-social-sciences-into-technology-degree_content.txt'}
Content preview:
SUTD Broadens Scope of Its Flagship Design and AI Degree, Becoming First University to Integrate Social Sciences Into a Technology Degree
SUTD Broadens Scope of Its Flagship Design and AI Degree, Becoming First University to Integrate Social Sciences Into a Technology Degree
DAI
HASS
DATE

In [34]:
# Execute the below query against the model and let it it answer from it's internal memory

query = "What courses are available in SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path(".env"))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# use same parameters as assignment 1
def openai_text(prompt: str, model: str = "gpt-4o-mini", max_output_tokens: int = 300, temperature: float = 0.2) -> str:
    resp = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
    )
    return resp.output_text

base_prompt = f"""You are a helpful assistant answering questions about SUTD.

Answer the question as accurately as possible using only your own internal knowledge.
If you are unsure, say you are unsure, do not fabricate something.

Question: {query}

Answer:"""

base_answer = openai_text(base_prompt, model="gpt-4o-mini", max_output_tokens=200, temperature=0.2)

print("Question:", query)
print("\nBase model answer (without RAG):\n")
print(base_answer)

#------------------------------


Question: What courses are available in SUTD?

Base model answer (without RAG):

Singapore University of Technology and Design (SUTD) offers a variety of courses across its different pillars and programs. The main areas of study include:

1. **Architecture and Sustainable Design (ASD)**
2. **Engineering Product Development (EPD)**
3. **Engineering Systems and Design (ESD)**
4. **Information Systems Technology and Design (ISTD)**

Each of these pillars includes core courses, electives, and interdisciplinary options that focus on design, technology, and innovation. Additionally, SUTD emphasizes hands-on learning and project-based work throughout its curriculum.

For the most accurate and up-to-date information on specific courses, it's best to refer to the official SUTD website or academic catalog.


In [41]:
# QUESTION: Now put everything together. Use langchain to integrate your vector store and Llama model into a RAG system
# Run the below example question against your RAG system.

# Example questions
query = "How can I increase my chances of admission into SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---


#------------------------------


In [36]:
# langchain llm wrapper

from langchain_core.language_models.llms import LLM
from typing import Optional, List

class OpenAIWrapper(LLM):
    model_name: str = "gpt-4o-mini"
    max_output_tokens: int = 300
    temperature: float = 0.2

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        return openai_text(
            prompt=prompt,
            model=self.model_name,
            max_output_tokens=self.max_output_tokens,
            temperature=self.temperature,
        )

    @property
    def _llm_type(self) -> str:
        return "openai_wrapper"

In [37]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant answering questions about SUTD.

Use the retrieved context below to answer the question as accurately as possible.
If the answer is partially available, provide the best supported answer based on the context.
If the answer is truly not present, say that the information is not available in the provided documents.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [38]:
# create retriever, llm, parser

from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

llm = OpenAIWrapper(
    model_name="gpt-4o-mini",
    max_output_tokens=300,
    temperature=0.2
)

output_parser = StrOutputParser()

In [39]:
# build llm langchain rag

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | rag_prompt
    | llm
    | output_parser
)

In [42]:
rag_answer = rag_chain.invoke(query)

print("Question:", query)
print("\nRAG answer:\n")
print(rag_answer)

Question: How can I increase my chances of admission into SUTD?

RAG answer:

To increase your chances of admission into SUTD, consider the following strategies:

1. **Strong Academic Performance**: Focus on achieving high grades, particularly in Mathematics (including Additional Mathematics) and Science subjects (Physics and/or Chemistry). SUTD looks for strong results in these areas.

2. **Relevant Qualifications**: If you have a diploma, ensure it is from a relevant field such as Engineering, Information Technology, Architecture, or Sciences, as these are assessed more favorably. Other diplomas may still be considered, but on a case-by-case basis.

3. **Holistic Application**: SUTD assesses applicants holistically, so highlight not just your academic achievements but also your qualities and traits. Demonstrate a growth mindset, collaboration skills, curiosity, and self-motivation.

4. **Portfolio and Extracurricular Activities**: Include any relevant projects, co-curricular activiti

In [43]:

# QUESTION: Below is set of test questions. Add another 10 test questions based on your user interviews and your value proposition canvas.
# Run the complete set of test questions against the RAG question answering system. 

questions = ["What are the admissions deadlines for SUTD?",
             "Is there financial aid available?",
             "What is the minimum score for the Mother Tongue Language?",
             "Do I require reference letters?",
             "Can polytechnic diploma students apply?",
             "Do I need SAT score?",
             "How many PhD students does SUTD have?",
             "How much are the tuition fees for Singaporeans?",
             "How much are the tuition fees for international students?",
             "Is there a minimum CAP?",
# added 10 here
            "What do I actually need to submit for my portfolio?",
            "Is it possible to get into SUTD with a polytechnic diploma?",
            "When will I find out if I got an interview?",
            "How are roommates matched in the hostel?",
            "What does the school like out for in applicants?",
            "How much does it cost to study there per year?",
            "If I don't have a good MTL grade, can I still get in?",
            "Does SUTD admissions rely solely on academic results?",
            "What exactly is the Computer Science course in SUTD about?",
            "What if my English is not good?"

            # added in qns from ass1 for testing. commented out aft refinement
            # "What subjects are common to all Freshmore students?",
            # "How are grades structured in Term 1 of the Freshmore curriculum?",
            # "What is the purpose of the computational thinking courses?",
            # "What is emphasized in the Introduction to Human-Centred Design course?",
            # "What are the two versions of the Math course offered?",
            # "What is the significance of the interdisciplinary design projects?",
            # "What subjects are included in Term 1 of the Freshmore curriculum?",
            # "What is the grading structure for subjects in Term 1?",
            # "What computational thinking courses are offered in the Freshmore curriculum?",
            # "What is the focus of the Design Computation course?",
            # "What are the two versions of the Math course offered in Terms 2 and 3?",
            # "What is the significance of hands-on projects in the Freshmore curriculum?",
            # "What is emphasized in the Global Humanities course?",
            # "What subjects are included in Term 2 of the Freshmore curriculum?",
            # "What foundational subjects are included in the Freshmore curriculum?",
            # "What is the focus of the Introduction to Programming course?",
            # "What are the two versions of the Math course offered in Term 2?",
            # "What is the focus of the Introduction to Human-Centred Design course?",
            # "What are the core subjects in Term 1 of the Freshmore curriculum?",
            # "What elective courses can students choose in Term 3?",
            # "What are the two versions of the Math course offered in Term 3?",
            # "What are the core subjects in Term 2 of the Freshmore curriculum?",
            # "What type of room are Freshmore students assigned to?",
            # "How are roommates matched in the hostel?",
            # "What security measures are in place at the hostel?",
            # "What amenities are available in the communal kitchen?",
            # "Where can students do laundry in the hostel?",
            # "What recreational facilities are available in the hostel?",
            # "Is there internet access in the hostel?",
            # "What is the purpose of live-in staff and student leaders?",
            # "What facilities are available on each floor of the hostel?",
            # "What is included in the communal kitchen on level 1?",
            # "What amenities are available in the pantry on level 9?",
            # "Where is the laundry room located?",
            # "What role do live-in staff and student leaders play?"
        ]
             

#--- ADD YOUR SOLUTION HERE (20 points)---
#---------------------------



In [46]:
import pandas as pd

results = []

for q in questions:
    retrieved_docs = retriever.invoke(q)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    rag_prompt_text = rag_prompt.format(
        context=context,
        question=q
    )

    # get model to output d answer
    answer = rag_chain.invoke(q)

    # make into table
    results.append({
        "question": q,
        "answer": answer,
        "top_source_1": retrieved_docs[0].metadata.get("doc_id", "") if len(retrieved_docs) > 0 else "",
        "top_source_1_text": retrieved_docs[0].page_content if len(retrieved_docs) > 0 else "",
        "top_source_2": retrieved_docs[1].metadata.get("doc_id", "") if len(retrieved_docs) > 1 else "",
        "top_source_2_text": retrieved_docs[1].page_content if len(retrieved_docs) > 1 else "",
        "top_source_3": retrieved_docs[2].metadata.get("doc_id", "") if len(retrieved_docs) > 2 else "",
        "top_source_3_text": retrieved_docs[2].page_content if len(retrieved_docs) > 2 else "",
    })

results_df = pd.DataFrame(results)
results_df.to_csv("data/rag_test_results.csv", index=False)
results_df

,question,answer,top_source_1,top_source_1_text,top_source_2,top_source_2_text,top_source_3,top_source_3_text
0,What are the admissions deadlines for SUTD?,The application period for local Diploma appli...,admissions_undergraduate_local-diploma_criteri...,"Most have strong results in STEM subjects, as ...",admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...,admissions_undergraduate_local-diploma_applica...,Your grades in the advanced modules will also ...
1,Is there financial aid available?,"Yes, there is financial aid available at SUTD....",admissions_undergraduate_education-expenses_co...,financial estimates\n.\nFor further informatio...,admissions_undergraduate_education-expenses_fi...,…\nUndergraduate admissions\nEducation expense...,admissions_undergraduate_education-expenses_co...,Admissions\nUndergraduate admissions\nEducatio...
2,What is the minimum score for the Mother Tongu...,The minimum score for the Mother Tongue Langua...,admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...,admissions_undergraduate_admission-requirement...,English Proficiency requirement\nSubmission of...,admissions_undergraduate_admission-requirement...,IELTS\nFor more information and registration f...
3,Do I require reference letters?,"Yes, you are required to provide reference let...",admissions_undergraduate_application-guide_con...,UPLOAD\nAcademic Transcripts and Certificates ...,admissions_undergraduate_admission-requirement...,"a pass in MTL A: Literature, or MTL A: Languag...",admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...
4,Can polytechnic diploma students apply?,"Yes, polytechnic diploma students can apply to...",admissions_undergraduate_local-diploma_criteri...,…\nUndergraduate admissions\nLocal Diploma\nCr...,admissions_undergraduate_local-diploma_applica...,…\nUndergraduate admissions\nLocal Diploma\nAp...,admissions_undergraduate_local-diploma_criteri...,",\nArchitecture\nand\nSciences\nare most relev..."
5,Do I need SAT score?,The SAT score is optional for admission to SUT...,admissions_undergraduate_admission-requirement...,English Proficiency requirement\nSubmission of...,admissions_undergraduate_local-diploma_criteri...,"Most have strong results in STEM subjects, as ...",admissions_undergraduate_admission-requirement...,IELTS\nFor more information and registration f...
6,How many PhD students does SUTD have?,The information is not available in the provid...,admissions_undergraduate_local-diploma_applica...,Your grades in the advanced modules will also ...,admissions_undergraduate_local-diploma_criteri...,Your grades in the advanced modules will also ...,admissions_undergraduate_admission-requirement...,…\nUndergraduate admissions\nAdmission require...
7,How much are the tuition fees for Singaporeans?,The tuition fees for Singapore Citizens (SC) a...,admissions_undergraduate_education-expenses_fe...,Tuition fees for\nexisting\nstudents admitted ...,admissions_undergraduate_education-expenses_fe...,…\nEducation expenses\nTuition fees and tuitio...,admissions_undergraduate_education-expenses_fe...,"(Inclusive of GST)\nPer academic year\nS$13,50..."
8,How much are the tuition fees for internationa...,The tuition fees for international students at...,admissions_undergraduate_education-expenses_fe...,"S$30,929\nAcademic Year 2020\nTuition fees for...",admissions_undergraduate_education-expenses_fe...,"(Inclusive of GST)\nPer academic year\nS$13,50...",admissions_undergraduate_education-expenses_fe...,…\nEducation expenses\nTuition fees and tuitio...
9,Is there a minimum CAP?,The information regarding a minimum CAP (Cumul...,istd_education_undergraduate_curriculum_content,"Electives (96 Credits)\nElectives, provide the...",esd_education_undergraduate_curriculum_beyond-...,By the numbers\nESD Core (84 Credits)\nThe ESD...,esd_education_undergraduate_curriculum_beyond-...,"HASS (60 Credits)\

### QUESTION: 


Manually inspect each answer, fact check whether the answer is correct (use Google or any other method) and check the retrieved documents

For each question, answer and context triple, record the following

- How accurate is the answer (1-5, 5 best)?
- How relevant is the retrieved context (1-5, 5 best)?
- How grounded is the answer in the retrieved context (instead of relying on the LLM's internal knowledge) (1-5, 5 best)?

**--- ADD YOUR SOLUTION HERE (20 points) ---**


My ratings are included in the excel sheet found at `data/rag_test_results annotated.xlsx`.
There were 2 results that stod out to me, rows 11 and 14.

For row 11:

**Question: Is there a minimum CAP?**

Answer: Yes, there is a minimum Cumulative Average Point (CAP) requirement. From AY2017 onwards, students must maintain a CAP of 3.5 or above to be eligible to enroll for an additional course in Term 5, 6, or 7.

The answer was inaccurate due to the wrong document being retrieved, but the groundedness of the answer was good. There is indeed a minimum CAP of 3.5 but that is not for admission into SUTD, that is for enrolling in additional courses in Terms 5,6 or 7.

This was the only one of its kind (where answer and retrieval was wrong, groundedness was good)

For row 14: 

**Question: When will I find out if I got an interview?**

Answer: You will be notified if selected for an interview (in-person or online conversation session) by the end of April.

The answer is actually accurate but the documents retrieved are wrong and even in the wrongly retrieved content nowhere does it mention April. Perhaps the model got the answer right by chance or by its own knowledge.

This was also the only one of its kind.

------------------------------

Impt note: My declaration of AI use can be found in the document named `AI Use Declaration.docx`



You can try improve the chatbot by going back to previous steps in the notebook and change things until the submission deadline. For example, you can add more data sources, change the embedding models, change the data pre-processing, etc. 


# End

This concludes Individual assignment 2.

Please submit this notebook with your answers and the generated output cells as a **Jupyter notebook file** via github.

1. Create a private github repository **sutd_5055mlop** under your github user.
2. Add your instructors as collaborator: ddahlmeier, bearwithchris and MarkHershey
3. Save your submission as `individual_assignment_02_StudentID`.ipynb 
4. Push the submission files to your repo 
5. Submit the link to the repo via eDimensions


